# Market-Specific Robustness — Literature-Motivated Main Specification

This notebook reruns the robustness analysis separately for:

1. `BTC_GOLD_2025`
2. `BTC_GOLD_2026`
3. `BTC_GOLD_SP500`

using the corrected YES/NO trading dataset.

## Models
- Conditional Logit
- Logistic Regression
- Random Forest
- XGBoost

## Baseline
- `Lag1_Return`
- `Lag2_Return`
- `Lag3_Return`

## Main Enhanced Polymarket block
- `PM_Prob_Lag1`
- `PM_Change_1D_Lag1`
- `PM_Prob_x_TimeToResolution`
- `PM_Log_Cumulative_Support_Notional_Lag1`
- `PM_Net_Support_Imbalance_Lag1`

Traditional ML additionally uses `Days_to_Resolution_Years`.

The specification is fixed from literature-motivated information channels rather than selected from final-test results.

Each market retains its own chronological 80/20 split and four expanding-window validation folds.


## Step 1 — Upload the Corrected Model-Ready Dataset

Upload:

`Final_Competitive_Markets_MODEL_READY_with_CORRECTED_PM_Trading_Features.xlsx`


## Step 2 — Install and Import Packages


In [ ]:
# ============================================================
# STEP 2: Install and Import Packages
# ============================================================
!pip -q install xgboost openpyxl statsmodels scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
from statsmodels.discrete.conditional_models import ConditionalLogit
import statsmodels.api as sm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
print("Packages imported successfully.")


## Step 3 — Load and Check the Data


In [ ]:
# ============================================================
# STEP 3: Load Corrected Model-Ready Master Sheet
# ============================================================

import glob
import os

candidates = glob.glob(
    "/content/*CORRECTED_PM_Trading_Features*.xlsx"
)

dataset_candidates = [
    x for x in candidates
    if "MODEL_READY" in os.path.basename(x).upper()
]

if len(dataset_candidates) != 1:
    raise ValueError(
        "Upload exactly one corrected model-ready dataset. "
        f"Detected: {dataset_candidates}"
    )

FILE_PATH = dataset_candidates[0]

df = pd.read_excel(
    FILE_PATH,
    sheet_name="Model_Ready_Master"
)

for col in [
    "Date",
    "Next_Financial_Date",
    "Financial_Prev1_Date",
    "Financial_Prev2_Date",
    "Financial_Prev3_Date",
    "PM_Signal_Date_Lag1",
    "PM_Refined_Activity_Signal_Date_Lag1"
]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

df = df.sort_values(
    ["Market_ID", "Date", "Event_ID", "Asset"]
).reset_index(drop=True)

print("Loaded:", FILE_PATH)
print("Rows:", len(df))
print("Events:", df["Event_ID"].nunique())

print("\nEvents by market:")
print(df.groupby("Market_ID")["Event_ID"].nunique())

print("\nDate ranges:")
display(
    df.groupby("Market_ID").agg(
        First_Date=("Date","min"),
        Last_Date=("Date","max"),
        Unique_Dates=("Date","nunique")
    )
)


## Step 4 — Define the New Feature Sets

The Baseline is unchanged. The Enhanced specification now combines the original probability features with the corrected trading-information core.


In [ ]:
BASELINE_FEATURES = [
    "Lag1_Return",
    "Lag2_Return",
    "Lag3_Return"
]

PM_MAIN_FEATURES = [
    "PM_Prob_Lag1",
    "PM_Change_1D_Lag1",
    "PM_Prob_x_TimeToResolution",
    "PM_Log_Cumulative_Support_Notional_Lag1",
    "PM_Net_Support_Imbalance_Lag1"
]

CL_ENHANCED_FEATURES = (
    BASELINE_FEATURES
    + PM_MAIN_FEATURES
)

ML_ENHANCED_FEATURES = (
    CL_ENHANCED_FEATURES
    + ["Days_to_Resolution_Years"]
)

RQ2_ONLY_FEATURES = [
    "PM_Change_From_Start_Lag1",
    "PM_Log_Support_Notional_Lag1",
    "PM_Relative_Support_Volume_Share_Lag1",
    "PM_Relative_Cumulative_Support_Share_Lag1",
    "PM_Log_Support_Unique_Takers_Lag1",
    "PM_Support_Volume_Surge_Lag1",
    "PM_Prob_x_Log_Contract_Notional_Lag1",
    "PM_Support_HHI_Lag1",
    "PM_Log_Support_Mean_Trade_Notional_Lag1",
    "PM_Candidate_Price_Dispersion_Lag1"
]

feature_set_table = pd.DataFrame([
    {
        "Specification": "Baseline",
        "Model_Family": "All",
        "Features": " + ".join(BASELINE_FEATURES)
    },
    {
        "Specification": "Enhanced",
        "Model_Family": "Conditional Logit",
        "Features": " + ".join(CL_ENHANCED_FEATURES)
    },
    {
        "Specification": "Enhanced",
        "Model_Family": "Traditional ML",
        "Features": " + ".join(ML_ENHANCED_FEATURES)
    }
])

print("Baseline:", BASELINE_FEATURES)
print("\nMain Polymarket features:", PM_MAIN_FEATURES)
print("\nConditional Logit Enhanced:", CL_ENHANCED_FEATURES)
print("\nTraditional ML Enhanced:", ML_ENHANCED_FEATURES)
print("\nRQ2 / robustness-only features:", RQ2_ONLY_FEATURES)


## Step 5 — Final Leakage and Structural Checks


In [ ]:
# ============================================================
# STEP 5: Leakage and Structure Checks
# ============================================================
winner_check = df.groupby("Event_ID")["Winner"].sum()
assert (winner_check == 1).all(), "ERROR: Some events do not have exactly one winner."
choice_size = df.groupby("Event_ID")["Asset"].nunique()
assert choice_size.isin([2,3]).all(), "ERROR: Unexpected choice-set size."
assert (df["PM_Signal_Date_Lag1"] < df["Date"]).all(), "ERROR: Polymarket timing leakage."
assert (
    df["PM_Refined_Activity_Signal_Date_Lag1"] < df["Date"]
).all(), "ERROR: Corrected PM trading-activity timing leakage."
assert (df["Financial_Prev1_Date"] < df["Date"]).all(), "ERROR: Financial lag timing leakage."
all_features = list(dict.fromkeys(BASELINE_FEATURES + CL_ENHANCED_FEATURES + ML_ENHANCED_FEATURES))
assert df[all_features].notna().all().all(), "ERROR: Missing feature values."
assert len(df) == 1012, "ERROR: Corrected master should contain 1,012 rows."
assert df["Event_ID"].nunique() == 428, "ERROR: Corrected master should contain 428 events."
print("All checks PASSED — including corrected PM trading timing.")


## Step 6 — Create a Separate Temporal 80/20 Split for Each Market


In [ ]:
# ============================================================
# STEP 6: Create Market-Specific 80/20 Splits
# ============================================================
MARKETS = ["BTC_GOLD_2025", "BTC_GOLD_2026", "BTC_GOLD_SP500"]
market_splits = {}

for market_id in MARKETS:
    m = df[df["Market_ID"] == market_id].copy()
    dates = np.array(sorted(m["Date"].unique()))
    cut = int(np.floor(0.80 * len(dates)))
    dev_dates, test_dates = dates[:cut], dates[cut:]
    market_splits[market_id] = {"Dev_Dates":dev_dates, "Test_Dates":test_dates}
    dev_m = m[m["Date"].isin(dev_dates)]
    test_m = m[m["Date"].isin(test_dates)]
    print("\n" + "="*75)
    print(market_id)
    print("="*75)
    print("Total dates:", len(dates))
    print("Development dates:", len(dev_dates))
    print("Final test dates:", len(test_dates))
    print("Development:", dev_m["Date"].min().date(), "to", dev_m["Date"].max().date(), "| Events:", dev_m["Event_ID"].nunique())
    print("Final test:", test_m["Date"].min().date(), "to", test_m["Date"].max().date(), "| Events:", test_m["Event_ID"].nunique())


## Step 7 — Build Four Expanding-Window Folds Within Each Market


In [ ]:
# ============================================================
# STEP 7: Build Market-Specific Expanding-Window Folds
# ============================================================
def build_market_folds(dev_dates, n_splits=4):
    n = len(dev_dates)
    test_size = max(8, int(np.floor(0.10 * n)))
    while n_splits * test_size >= int(0.60 * n):
        test_size -= 1
    if test_size < 5:
        raise ValueError("Not enough dates for expanding-window folds.")
    splitter = TimeSeriesSplit(n_splits=n_splits, test_size=test_size, gap=0)
    folds=[]
    for fold_no,(tr,va) in enumerate(splitter.split(dev_dates), start=1):
        folds.append({"Fold":fold_no,"Train_Dates":dev_dates[tr],"Val_Dates":dev_dates[va]})
    return folds

market_folds = {}
for market_id in MARKETS:
    folds = build_market_folds(market_splits[market_id]["Dev_Dates"], 4)
    market_folds[market_id] = folds
    print("\n" + "="*75)
    print("FOLDS:", market_id)
    print("="*75)
    for f in folds:
        print(f"Fold {f['Fold']}:", pd.Timestamp(f['Train_Dates'][0]).date(), "to", pd.Timestamp(f['Train_Dates'][-1]).date(), "-> validation", pd.Timestamp(f['Val_Dates'][0]).date(), "to", pd.Timestamp(f['Val_Dates'][-1]).date())


## Step 8 — Event-Level Evaluation Metrics


In [ ]:
# ============================================================
# STEP 8: Event-Level Evaluation Function
# ============================================================
def evaluate_event_predictions(pred):
    eps=1e-15
    rows=[]
    for event_id,g in pred.groupby("Event_ID"):
        probs=g["Pred_Prob"].to_numpy(float)
        probs=probs/probs.sum()
        y=g["Winner"].to_numpy(int)
        actual_idx=np.where(y==1)[0][0]
        pred_idx=int(np.argmax(probs))
        actual=g.iloc[actual_idx]["Asset"]
        predicted=g.iloc[pred_idx]["Asset"]
        winner_prob=float(np.clip(probs[actual_idx],eps,1-eps))
        J=len(g)
        rows.append({
            "Market_ID":g["Market_ID"].iloc[0],
            "Event_ID":event_id,
            "Date":g["Date"].iloc[0],
            "Choice_Set_Size":J,
            "Actual_Winner":actual,
            "Predicted_Winner":predicted,
            "Winner_Probability":winner_prob,
            "Correct":int(actual==predicted),
            "Log_Loss":-np.log(winner_prob),
            "Brier_Score":np.sum((probs-y)**2),
            "Uniform_Log_Loss":np.log(J),
            "Uniform_Brier":1-(1/J)
        })
    events=pd.DataFrame(rows)
    summary={
        "Events":len(events),
        "Log_Loss":events["Log_Loss"].mean(),
        "Brier_Score":events["Brier_Score"].mean(),
        "Accuracy":events["Correct"].mean(),
        "Chance_Accuracy":(1/events["Choice_Set_Size"]).mean(),
        "Log_Skill_vs_Uniform":1-(events["Log_Loss"].sum()/events["Uniform_Log_Loss"].sum()),
        "Brier_Skill_vs_Uniform":1-(events["Brier_Score"].sum()/events["Uniform_Brier"].sum())
    }
    return summary,events


## Step 9 — Conditional Logit Function


In [ ]:
# ============================================================
# STEP 9: Conditional Logit Fit and Predict
# ============================================================
def fit_clogit_predict(train_df,test_df,features):
    scaler=StandardScaler()
    X_train=scaler.fit_transform(train_df[features])
    X_test=scaler.transform(test_df[features])
    y=train_df["Winner"].astype(int).to_numpy()
    groups=train_df["Event_ID"].astype(str).to_numpy()
    model=ConditionalLogit(y,X_train,groups=groups)
    result=model.fit(method="bfgs",maxiter=1000,disp=False)
    utility=X_test @ np.asarray(result.params)
    pred=test_df[["Market_ID","Event_ID","Date","Asset","Winner"]].copy()
    pred["Utility"]=utility
    pred["Pred_Prob"]=np.nan
    for event_id,idx in pred.groupby("Event_ID").groups.items():
        idx=list(idx)
        pred.loc[idx,"Pred_Prob"]=softmax(pred.loc[idx,"Utility"].to_numpy())
    return result,pred


## Step 10 — Traditional ML Functions


In [ ]:
# ============================================================
# STEP 10A: Define ML Grids and Model Builder
# ============================================================
MODEL_GRIDS={
    "Logistic_Regression":[{"C":0.1},{"C":1.0},{"C":10.0}],
    "Random_Forest":[
        {"n_estimators":300,"max_depth":3,"min_samples_leaf":5,"max_features":"sqrt"},
        {"n_estimators":300,"max_depth":5,"min_samples_leaf":5,"max_features":"sqrt"},
        {"n_estimators":500,"max_depth":None,"min_samples_leaf":5,"max_features":"sqrt"}
    ],
    "XGBoost":[
        {"n_estimators":150,"max_depth":2,"learning_rate":0.03,"subsample":0.8,"colsample_bytree":0.8},
        {"n_estimators":200,"max_depth":3,"learning_rate":0.03,"subsample":0.8,"colsample_bytree":0.8},
        {"n_estimators":150,"max_depth":3,"learning_rate":0.05,"subsample":0.8,"colsample_bytree":0.8}
    ]
}

def build_model(model_name,params):
    if model_name=="Logistic_Regression":
        return Pipeline([("scaler",StandardScaler()),("model",LogisticRegression(C=params["C"],penalty="l2",solver="lbfgs",max_iter=2000,random_state=42))])
    if model_name=="Random_Forest":
        return RandomForestClassifier(n_estimators=params["n_estimators"],max_depth=params["max_depth"],min_samples_leaf=params["min_samples_leaf"],max_features=params["max_features"],random_state=42,n_jobs=-1)
    if model_name=="XGBoost":
        return XGBClassifier(n_estimators=params["n_estimators"],max_depth=params["max_depth"],learning_rate=params["learning_rate"],subsample=params["subsample"],colsample_bytree=params["colsample_bytree"],objective="binary:logistic",eval_metric="logloss",random_state=42,n_jobs=-1,tree_method="hist",verbosity=0)
    raise ValueError("Unknown model")


In [ ]:
# ============================================================
# STEP 10B: Convert ML Row Scores into Competitive Probabilities
# ============================================================
def ml_event_predictions(model,test_df,features):
    raw=model.predict_proba(test_df[features])[:,1]
    pred=test_df[["Market_ID","Event_ID","Date","Asset","Winner"]].copy()
    pred["Raw_Winner_Prob"]=raw
    pred["Pred_Prob"]=np.nan
    for event_id,idx in pred.groupby("Event_ID").groups.items():
        idx=list(idx)
        probs=pred.loc[idx,"Raw_Winner_Prob"].to_numpy(float)
        probs=np.clip(probs,1e-15,None)
        probs=probs/probs.sum()
        pred.loc[idx,"Pred_Prob"]=probs
    return pred


## Step 11 — Run Conditional Logit Separately for Each Market


In [ ]:
# ============================================================
# STEP 11: Conditional Logit Market-Specific Validation and Test
# ============================================================
cl_cv_rows=[]
cl_final_rows=[]
cl_final_events=[]
cl_coefficients=[]

for market_id in MARKETS:
    market_df=df[df["Market_ID"]==market_id].copy()
    dev_dates=market_splits[market_id]["Dev_Dates"]
    test_dates=market_splits[market_id]["Test_Dates"]
    folds=market_folds[market_id]
    print("\n"+"="*80)
    print("CONDITIONAL LOGIT:",market_id)
    print("="*80)
    for specification,features in [("Baseline",BASELINE_FEATURES),("Enhanced",CL_ENHANCED_FEATURES)]:
        for fold in folds:
            tr=market_df[market_df["Date"].isin(fold["Train_Dates"])]
            va=market_df[market_df["Date"].isin(fold["Val_Dates"])]
            result,pred=fit_clogit_predict(tr,va,features)
            metrics,_=evaluate_event_predictions(pred)
            metrics.update({"Model":"Conditional_Logit","Market_ID":market_id,"Specification":specification,"Fold":fold["Fold"]})
            cl_cv_rows.append(metrics)
    dev_df=market_df[market_df["Date"].isin(dev_dates)]
    test_df=market_df[market_df["Date"].isin(test_dates)]
    for specification,features in [("Baseline",BASELINE_FEATURES),("Enhanced",CL_ENHANCED_FEATURES)]:
        result,pred=fit_clogit_predict(dev_df,test_df,features)
        metrics,events=evaluate_event_predictions(pred)
        metrics.update({"Model":"Conditional_Logit","Market_ID":market_id,"Specification":specification})
        cl_final_rows.append(metrics)
        events["Model"]="Conditional_Logit"
        events["Specification"]=specification
        cl_final_events.append(events)
        for feature,coef,pval in zip(features,np.asarray(result.params),np.asarray(result.pvalues)):
            cl_coefficients.append({"Market_ID":market_id,"Specification":specification,"Feature":feature,"Coefficient_Standardized":coef,"P_Value":pval})

cl_cv_results=pd.DataFrame(cl_cv_rows)
cl_final_results=pd.DataFrame(cl_final_rows)
cl_final_event_predictions=pd.concat(cl_final_events,ignore_index=True)
cl_coefficient_table=pd.DataFrame(cl_coefficients)
display(cl_final_results)


## Step 12 — Tune Traditional ML Separately Within Each Market


In [ ]:
# ============================================================
# STEP 12: Market-Specific ML Hyperparameter Tuning
# ============================================================
ml_tuning_rows=[]
for market_id in MARKETS:
    market_df=df[df["Market_ID"]==market_id].copy()
    folds=market_folds[market_id]
    print("\n"+"="*80)
    print("ML TUNING:",market_id)
    print("="*80)
    for model_name,grid in MODEL_GRIDS.items():
        for specification,features in [("Baseline",BASELINE_FEATURES),("Enhanced",ML_ENHANCED_FEATURES)]:
            for candidate_no,params in enumerate(grid,start=1):
                logs=[]; briers=[]; accs=[]
                for fold in folds:
                    tr=market_df[market_df["Date"].isin(fold["Train_Dates"])]
                    va=market_df[market_df["Date"].isin(fold["Val_Dates"])]
                    model=build_model(model_name,params)
                    model.fit(tr[features],tr["Winner"].astype(int))
                    pred=ml_event_predictions(model,va,features)
                    metrics,_=evaluate_event_predictions(pred)
                    logs.append(metrics["Log_Loss"]); briers.append(metrics["Brier_Score"]); accs.append(metrics["Accuracy"])
                ml_tuning_rows.append({"Market_ID":market_id,"Model":model_name,"Specification":specification,"Candidate":candidate_no,"Parameters":str(params),"Mean_CV_Log_Loss":np.mean(logs),"Mean_CV_Brier":np.mean(briers),"Mean_CV_Accuracy":np.mean(accs)})
ml_tuning=pd.DataFrame(ml_tuning_rows)
display(ml_tuning.sort_values(["Market_ID","Model","Specification","Mean_CV_Log_Loss"]))


## Step 13 — Select Best ML Hyperparameters


In [ ]:
# ============================================================
# STEP 13: Select Best ML Hyperparameters
# ============================================================
best_ml_rows=(ml_tuning.sort_values("Mean_CV_Log_Loss").groupby(["Market_ID","Model","Specification"],as_index=False).first())
display(best_ml_rows)
best_ml_params={}
for _,r in best_ml_rows.iterrows():
    best_ml_params[(r["Market_ID"],r["Model"],r["Specification"])]=MODEL_GRIDS[r["Model"]][int(r["Candidate"])-1]


## Step 14 — Run Selected ML Models on Validation Folds


In [ ]:
# ============================================================
# STEP 14: Selected Market-Specific ML Cross-Validation
# ============================================================
ml_cv_rows=[]
for market_id in MARKETS:
    market_df=df[df["Market_ID"]==market_id].copy()
    for model_name in MODEL_GRIDS.keys():
        for specification,features in [("Baseline",BASELINE_FEATURES),("Enhanced",ML_ENHANCED_FEATURES)]:
            params=best_ml_params[(market_id,model_name,specification)]
            for fold in market_folds[market_id]:
                tr=market_df[market_df["Date"].isin(fold["Train_Dates"])]
                va=market_df[market_df["Date"].isin(fold["Val_Dates"])]
                model=build_model(model_name,params)
                model.fit(tr[features],tr["Winner"].astype(int))
                pred=ml_event_predictions(model,va,features)
                metrics,_=evaluate_event_predictions(pred)
                metrics.update({"Model":model_name,"Market_ID":market_id,"Specification":specification,"Fold":fold["Fold"]})
                ml_cv_rows.append(metrics)
ml_cv_results=pd.DataFrame(ml_cv_rows)
display(ml_cv_results.head())


## Step 15 — Run Selected ML Models on Each Market's Final 20%


In [ ]:
# ============================================================
# STEP 15: Market-Specific Final ML Holdout Tests
# ============================================================
ml_final_rows=[]
ml_final_events=[]
ml_feature_importance_rows=[]
for market_id in MARKETS:
    market_df=df[df["Market_ID"]==market_id].copy()
    dev_df=market_df[market_df["Date"].isin(market_splits[market_id]["Dev_Dates"])]
    test_df=market_df[market_df["Date"].isin(market_splits[market_id]["Test_Dates"])]
    for model_name in MODEL_GRIDS.keys():
        for specification,features in [("Baseline",BASELINE_FEATURES),("Enhanced",ML_ENHANCED_FEATURES)]:
            params=best_ml_params[(market_id,model_name,specification)]
            model=build_model(model_name,params)
            model.fit(dev_df[features],dev_df["Winner"].astype(int))
            pred=ml_event_predictions(model,test_df,features)
            metrics,events=evaluate_event_predictions(pred)
            metrics.update({"Model":model_name,"Market_ID":market_id,"Specification":specification,"Parameters":str(params)})
            ml_final_rows.append(metrics)
            events["Model"]=model_name; events["Specification"]=specification
            ml_final_events.append(events)
            if specification=="Enhanced":
                if model_name=="Logistic_Regression":
                    values=model.named_steps["model"].coef_[0]; imp_type="Standardized coefficient"
                else:
                    values=model.feature_importances_; imp_type="Feature importance"
                for feature,value in zip(features,values):
                    ml_feature_importance_rows.append({"Market_ID":market_id,"Model":model_name,"Feature":feature,"Importance_or_Coefficient":value,"Type":imp_type})
ml_final_results=pd.DataFrame(ml_final_rows)
ml_final_event_predictions=pd.concat(ml_final_events,ignore_index=True)
ml_feature_importance=pd.DataFrame(ml_feature_importance_rows)
display(ml_final_results.sort_values(["Market_ID","Model","Specification"]))


## Step 16 — Combine All Four Models


In [ ]:
# ============================================================
# STEP 16: Combine Conditional Logit and ML Results
# ============================================================
all_final_results=pd.concat([cl_final_results,ml_final_results],ignore_index=True)
all_cv_results=pd.concat([cl_cv_results,ml_cv_results],ignore_index=True)
display(all_final_results.sort_values(["Market_ID","Model","Specification"]))


## Step 17 — Calculate Baseline vs Enhanced Differences


In [ ]:
# ============================================================
# STEP 17: Final Baseline vs Enhanced Differences
# ============================================================
difference_rows=[]
for (market_id,model_name),g in all_final_results.groupby(["Market_ID","Model"]):
    temp=g.set_index("Specification")
    if "Baseline" not in temp.index or "Enhanced" not in temp.index:
        continue
    difference_rows.append({
        "Market_ID":market_id,"Model":model_name,
        "Baseline_Log_Loss":temp.loc["Baseline","Log_Loss"],
        "Enhanced_Log_Loss":temp.loc["Enhanced","Log_Loss"],
        "LogLoss_Enhanced_minus_Baseline":temp.loc["Enhanced","Log_Loss"]-temp.loc["Baseline","Log_Loss"],
        "Baseline_Brier":temp.loc["Baseline","Brier_Score"],
        "Enhanced_Brier":temp.loc["Enhanced","Brier_Score"],
        "Brier_Enhanced_minus_Baseline":temp.loc["Enhanced","Brier_Score"]-temp.loc["Baseline","Brier_Score"],
        "Baseline_Accuracy":temp.loc["Baseline","Accuracy"],
        "Enhanced_Accuracy":temp.loc["Enhanced","Accuracy"],
        "Accuracy_Enhanced_minus_Baseline":temp.loc["Enhanced","Accuracy"]-temp.loc["Baseline","Accuracy"]
    })
final_differences=pd.DataFrame(difference_rows)
print("Negative Log Loss/Brier difference = Enhanced better.")
print("Positive Accuracy difference = Enhanced better.")
display(final_differences.sort_values(["Market_ID","Model"]))


## Step 18 — Cross-Validation Summary by Market and Model


In [ ]:
# ============================================================
# STEP 18: Market-Specific CV Summary
# ============================================================
cv_summary=(all_cv_results.groupby(["Market_ID","Model","Specification"]).agg(
    Mean_Log_Loss=("Log_Loss","mean"), SD_Log_Loss=("Log_Loss","std"),
    Mean_Brier=("Brier_Score","mean"), SD_Brier=("Brier_Score","std"),
    Mean_Accuracy=("Accuracy","mean"), SD_Accuracy=("Accuracy","std")
).reset_index())
display(cv_summary.sort_values(["Market_ID","Model","Specification"]))


## Step 19 — HAC Loss-Difference Tests Within Each Market


In [ ]:
# ============================================================
# STEP 19: HAC Tests Within Each Market and Model
# ============================================================
all_event_predictions=pd.concat([cl_final_event_predictions,ml_final_event_predictions],ignore_index=True)

def hac_test(series,maxlags=3):
    x=pd.Series(series).dropna().astype(float)
    X=np.ones((len(x),1))
    fit=sm.OLS(x.to_numpy(),X).fit(cov_type="HAC",cov_kwds={"maxlags":maxlags})
    return {"Mean_Difference":fit.params[0],"HAC_Std_Error":fit.bse[0],"t_Statistic":fit.tvalues[0],"p_Value":fit.pvalues[0],"N_Events":len(x)}

hac_rows=[]
for (market_id,model_name),g in all_event_predictions.groupby(["Market_ID","Model"]):
    wide=g.pivot(index="Event_ID",columns="Specification",values=["Log_Loss","Brier_Score"])
    log_diff=wide["Log_Loss"]["Baseline"]-wide["Log_Loss"]["Enhanced"]
    brier_diff=wide["Brier_Score"]["Baseline"]-wide["Brier_Score"]["Enhanced"]
    hac_rows.append({"Market_ID":market_id,"Model":model_name,"Metric":"Log Loss",**hac_test(log_diff)})
    hac_rows.append({"Market_ID":market_id,"Model":model_name,"Metric":"Brier Score",**hac_test(brier_diff)})
hac_results=pd.DataFrame(hac_rows)
print("Positive Mean_Difference means Enhanced has lower loss.")
display(hac_results.sort_values(["Market_ID","Model","Metric"]))


## Step 20 — Compact Market-Specific Summary for Felipe


In [ ]:
# ============================================================
# STEP 20: Compact Supervisor Summary
# ============================================================
felipe_rows=[]
for _,r in final_differences.iterrows():
    felipe_rows.append({
        "Market":r["Market_ID"],
        "Model":r["Model"],
        "Baseline Log Loss":r["Baseline_Log_Loss"],
        "Enhanced Log Loss":r["Enhanced_Log_Loss"],
        "Baseline Accuracy":r["Baseline_Accuracy"],
        "Enhanced Accuracy":r["Enhanced_Accuracy"],
        "Primary Result":"Improved" if r["LogLoss_Enhanced_minus_Baseline"]<0 else "No improvement"
    })
felipe_summary=pd.DataFrame(felipe_rows)
display(felipe_summary.sort_values(["Market","Model"]))


## Step 21 — Save All Market-Specific Robustness Results


In [ ]:
# ============================================================
# STEP 21: Save Results to Excel
# ============================================================
OUTPUT_FILE="/content/Market_Specific_Robustness_All_Models_Results.xlsx"
with pd.ExcelWriter(OUTPUT_FILE,engine="openpyxl") as writer:
    all_final_results.to_excel(writer,sheet_name="Final_Test_All_Models",index=False)
    final_differences.to_excel(writer,sheet_name="Final_Differences",index=False)
    all_cv_results.to_excel(writer,sheet_name="CV_Fold_Results",index=False)
    cv_summary.to_excel(writer,sheet_name="CV_Summary",index=False)
    ml_tuning.to_excel(writer,sheet_name="ML_Tuning_All",index=False)
    best_ml_rows.to_excel(writer,sheet_name="Best_ML_Params",index=False)
    cl_coefficient_table.to_excel(writer,sheet_name="CL_Coefficients",index=False)
    ml_feature_importance.to_excel(writer,sheet_name="ML_Feature_Importance",index=False)
    all_event_predictions.to_excel(writer,sheet_name="Final_Event_Predictions",index=False)
    hac_results.to_excel(writer,sheet_name="HAC_Tests",index=False)
    felipe_summary.to_excel(writer,sheet_name="Felipe_Summary",index=False)
    feature_set_table.to_excel(
        writer,
        sheet_name="Feature_Set_Definitions",
        index=False
    )

print("Saved successfully:")
print(OUTPUT_FILE)


## Step 22 — Download the Robustness Results


In [ ]:
# ============================================================
# STEP 22: Download Results Workbook
# ============================================================
from google.colab import files
files.download("/content/Market_Specific_Robustness_All_Models_Results.xlsx")
